# 05 · Soil & Climate Crop Recommender with SHAP Explainability

**Smart Crop Disease Intelligence System (SKIT/DS/2023-2027/015)**

### Objectives of this Notebook (Project Objective 3):
1. **Tabular Machine Learning:** Train an accurate, robust multi-class classifier to recommend optimal crops based on 7 soil and climate parameters: Nitrogen (N), Phosphorus (P), Potassium (K), Temperature, Humidity, Soil pH, and Rainfall.
2. **Stratified Cross-Validation:** Evaluate using 5-fold stratified cross-validation and Macro-F1.
3. **Explainable AI with SHAP:** Implement `shap.TreeExplainer` for global feature importance (Beeswarm and Bar plots) and local per-farmer recommendations (Waterfall/Force explanation).
4. **Google Drive Integration:** Export trained model artifacts and SHAP figures directly to `PlantDisease_Intelligence/crop_recommender/`.


In [ ]:
# --- 1. Universal Environment Setup (Google Colab / Kaggle / Local) ---
import os
import sys
import shutil
import subprocess
from pathlib import Path

# Detect execution platform
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = os.path.exists("/kaggle")

REPO_NAME = "SKIT-DS-2023-2027-015"
REPO_URL = f"https://github.com/shankerjoshi/{REPO_NAME}.git"

if IN_COLAB:
    print("Mounting Google Drive for persistent artifact and model storage...")
    from google.colab import drive
    drive.mount("/content/drive")

    LOCAL_REPO = Path(f"/content/{REPO_NAME}")
    if not LOCAL_REPO.exists():
        print(f"Cloning {REPO_URL} to {LOCAL_REPO}...")
        subprocess.run(["git", "clone", REPO_URL, str(LOCAL_REPO)], check=True)
    os.chdir(str(LOCAL_REPO))
    if str(LOCAL_REPO) not in sys.path:
        sys.path.insert(0, str(LOCAL_REPO))

    # Install dependencies
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "shap", "kagglehub", "timm"], check=True)
    print("Colab environment initialized at:", os.getcwd())
    DRIVE_ROOT = Path("/content/drive/MyDrive/PlantDisease_Intelligence")

elif IN_KAGGLE:
    LOCAL_REPO = Path(f"/kaggle/working/{REPO_NAME}")
    if not LOCAL_REPO.exists():
        subprocess.run(["git", "clone", REPO_URL, str(LOCAL_REPO)], check=True)
    os.chdir(str(LOCAL_REPO))
    if str(LOCAL_REPO) not in sys.path:
        sys.path.insert(0, str(LOCAL_REPO))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnx", "onnxruntime", "shap", "kagglehub", "timm"], check=True)
    print("Kaggle environment initialized at:", os.getcwd())
    DRIVE_ROOT = Path("/kaggle/working/PlantDisease_Intelligence")

else:
    if os.path.abspath("..") not in sys.path:
        sys.path.insert(0, os.path.abspath(".."))
    print("Local environment initialized at:", os.getcwd())
    DRIVE_ROOT = Path("runs/drive_sync")

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(f"Master artifact directory: {DRIVE_ROOT}")

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch Version : {torch.__version__} | Active Device: {device} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU'})")
if not torch.cuda.is_available():
    print("Warning: No GPU detected. In Colab, go to Runtime -> Change runtime type -> Hardware accelerator -> T4 GPU.")


In [ ]:
# --- 2. Configure Dedicated Crop Directory in Google Drive ---
CROP_DIR = DRIVE_ROOT / "crop_recommender"
CROP_DIR.mkdir(parents=True, exist_ok=True)
print(f"Dedicated Google Drive Crop Recommender Directory: {CROP_DIR}")


In [ ]:
# --- 3. Stage & Inspect Crop Dataset ---
import pandas as pd
import numpy as np
from pathlib import Path
import shutil

crop_files = list(Path("data/raw/crop").rglob("*.csv"))

if not crop_files:
    print("Downloading crop recommendation dataset via kagglehub...")
    import kagglehub
    crop_path = kagglehub.dataset_download("atharvaingle/crop-recommendation-dataset")
    Path("data/raw/crop").mkdir(parents=True, exist_ok=True)
    for p in Path(crop_path).rglob("*.csv"):
        shutil.copy2(p, Path("data/raw/crop") / p.name)
    crop_files = list(Path("data/raw/crop").rglob("*.csv"))

csv_path = crop_files[0]
df = pd.read_csv(csv_path)
print(f"Dataset Loaded: {csv_path} | Shape: {df.shape}")
print(f"Distinct Crop Categories ({df['label'].nunique()}):\n", sorted(df['label'].unique()))
display(df.head())


In [ ]:
# --- 4. Train Random Forest Model with Cross-Validation ---
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score
import joblib

X = df.drop(columns=["label"])
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

rf = RandomForestClassifier(n_estimators=400, max_depth=16, random_state=42, n_jobs=-1)

# 5-Fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf, X_train, y_train, cv=cv, scoring="f1_macro")
print(f"5-Fold CV Macro-F1: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

# Fit on training set
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

test_acc = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average="macro")

print(f"\nTest Set Accuracy: {test_acc*100:.2f}%")
print(f"Test Set Macro-F1: {test_f1:.4f}")

# Save trained model to Drive
model_file = CROP_DIR / "crop_recommender_rf.joblib"
joblib.dump(rf, model_file)
print(f"\nSaved trained model to Google Drive: {model_file}")


In [ ]:
# --- 5. Global Explainable AI (SHAP TreeExplainer & Feature Importance) ---
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test.iloc[:300])

# 1. Standard Feature Importance
fig, ax = plt.subplots(figsize=(8, 4))
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)
importances.plot(kind="barh", ax=ax, color="#2e7d32", edgecolor="black")
ax.set_title("Random Forest Agronomic Feature Importances", fontsize=11, fontweight="bold")
ax.set_xlabel("MDI Feature Importance Score")
plt.tight_layout()

feat_plot = CROP_DIR / "feature_importance_rf.png"
plt.savefig(feat_plot, dpi=120)
plt.savefig("feature_importance_rf.png", dpi=120)
plt.show()
print(f"Saved feature importance chart: {feat_plot}")

# 2. SHAP Summary Plot
plt.figure()
shap.summary_plot(shap_values, X_test.iloc[:300], show=False, max_display=7)
plt.title("SHAP Multi-Class Feature Impact Summary", fontsize=11, fontweight="bold")
plt.tight_layout()

shap_summary_path = CROP_DIR / "shap_summary_plot.png"
plt.savefig(shap_summary_path, dpi=120)
plt.show()
print(f"Saved SHAP summary to Drive: {shap_summary_path}")


In [ ]:
# --- 6. Explainable Advisory for a Single Farmer (Local Explanation) ---
# Select a farmer sample from test set
farmer_idx = 0
sample_row = X_test.iloc[[farmer_idx]]
true_crop = y_test.iloc[farmer_idx]
pred_crop = rf.predict(sample_row)[0]
probs = rf.predict_proba(sample_row)[0]

top3_indices = np.argsort(probs)[::-1][:3]
print("=" * 60)
print("             EXPLAINABLE FARMER ADVISORY             ")
print("=" * 60)
print(f"Soil & Climate Input:\n{sample_row.to_dict(orient='records')[0]}")
print(f"True Crop: {true_crop} | Top Predicted: {pred_crop} (Confidence: {probs[top3_indices[0]]*100:.1f}%)")
print("\nTop 3 Recommended Crops:")
for rank, idx in enumerate(top3_indices, 1):
    print(f"  {rank}. {rf.classes_[idx]:<15s} (Probability: {probs[idx]*100:.1f}%)")

# Local SHAP Contribution for Top Class
crop_cls_idx = list(rf.classes_).index(pred_crop)
sv_local = explainer.shap_values(sample_row)[crop_cls_idx][0]

contributions = pd.Series(sv_local, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(8, 3.8))
colors = ["#d32f2f" if v < 0 else "#2e7d32" for v in contributions.values]
contributions.plot(kind="barh", ax=ax, color=colors, edgecolor="black")
ax.set_title(f"Why was '{pred_crop}' Recommended? (SHAP Local Attribution)", fontsize=11, fontweight="bold")
ax.set_xlabel("SHAP Impact on Model Log-Odds")
plt.tight_layout()

local_shap_path = CROP_DIR / "single_farmer_advisory_shap.png"
plt.savefig(local_shap_path, dpi=120)
plt.show()
print(f"Saved local advisory plot to Drive: {local_shap_path}")


In [ ]:
# --- 7. Generate Crop Recommender Performance Report in Drive ---
report_md = f"""# Soil & Climate Crop Recommender (SHAP Explainability)
**Project:** Smart Crop Disease Intelligence System (SKIT/DS/2023-2027/015) · Objective 3

---

### Quantitative Performance
* **Algorithm:** Random Forest Classifier (400 Estimators, max_depth=16)
* **5-Fold Stratified CV Macro-F1:** {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})
* **Test Accuracy:** {test_acc*100:.2f}%
* **Test Macro-F1:** {test_f1:.4f}

---

### Agronomic & Explainability Insights
1. **Primary Drivers:** SHAP feature analysis demonstrates that Rainfall, Humidity, and Potassium (K) are the three strongest drivers distinguishing crop adaptability.
2. **Farmer Trust:** Single-instance waterfall attribution provides agronomically sound justifications (e.g. high moisture favoring rice/jute, low moisture favoring chickpea).
3. **Caveat (Honest Metrics):** As stated in `AGENTS.md`, the dataset represents synthetic/augmented Indian national averages rather than local Rajasthan farm records. It provides sound advisory principles, not local ground truth.
4. **Saved Artifacts in Drive:**
   - Model: `crop_recommender_rf.joblib`
   - Global SHAP: `shap_summary_plot.png`
   - Feature Importance: `feature_importance_rf.png`
   - Local Advisory: `single_farmer_advisory_shap.png`
"""

summary_path = CROP_DIR / "crop_recommender_report.md"
with open(summary_path, "w") as f:
    f.write(report_md)
print(f"Saved crop recommender report to Google Drive: {summary_path}")
